# 实验一：静态页面与 API 数据采集

**平台：SmartPark DataLab**（`https://115.159.159.82`）

本 Notebook 完成两类典型 Web 数据采集：

| 目标 | 加载方式 | 采集入口 | 采集量 |
|---|---|---|---|
| **Device** | 动态加载（JSON API） | `GET /api/devices`，用 `page` / `page_size` 分页 | 全部 **5000** 条 |
| **Company** | 静态加载（HTML） | `GET /companies?page=N` + BeautifulSoup + CSS Selector | 前 3 页共 **60** 条 |

> 说明：站点证书有效，`requests` 使用默认校验即可。若课堂环境提示证书错误，
> 可临时加 `session.verify = False`。

## 0. 环境准备

In [1]:
import re
import json
from pathlib import Path
from collections import Counter
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

BASE_URL = "https://115.159.159.82"

session = requests.Session()
session.headers.update({"User-Agent": "SmartPark-DataLab-Lab1/1.0"})

print("环境准备完成 ->", BASE_URL)

环境准备完成 -> https://115.159.159.82


# 任务 1：判断数据加载方式

## 1.1 Device：判断动态加载并找到 API

判断方法：把“右键 → 查看网页源代码”这一步用代码等价复现——直接取 `/devices` 返回的**初始 HTML**，
然后在里面搜索页面上明明已经看到的设备数据。

In [2]:
r = session.get(BASE_URL + "/devices", timeout=10)
html = r.text

print("status:", r.status_code)
print("初始 HTML 长度:", len(html))
print("源码中出现设备编号 D00000001 ?        ", "D00000001" in html)
print("源码中出现设备名 循环泵-000001-001 ?  ", "循环泵-000001-001" in html)
print("源码中出现 API 地址 /api/devices ?    ", "/api/devices" in html)
print()
# 页面上明明有设备表格，但初始 HTML 里表格是空的
i = html.find('id="device-rows"')
print("初始 HTML 中表格数据区片段：")
print("  ...", html[i - 30: i + 90], "...")
print()
print("页面真正用来取数据的脚本：")
print(" ", re.findall(r'<script src="[^"]+devices\.js"></script>', html))

status: 200
初始 HTML 长度: 2949
源码中出现设备编号 D00000001 ?         False
源码中出现设备名 循环泵-000001-001 ?   False
源码中出现 API 地址 /api/devices ?     False

初始 HTML 中表格数据区片段：
  ... th>状态</th></tr></thead><tbody id="device-rows"><tr><td colspan="8" class="loading-cell">正在加载…</td></tr></tbody></table>< ...

页面真正用来取数据的脚本：
  ['<script src="https://115.159.159.82/static/js/devices.js"></script>']


结论：

1. 初始 HTML 里**搜不到任何一条真实设备记录**（设备编号、设备名都搜不到）；
2. 表格数据区只有一行占位：`<tbody id="device-rows">…正在加载…</tbody>`；
3. 真正取数的脚本是 `/static/js/devices.js`，它内部执行 `fetch('/api/devices?...')`。

所以 `/devices` 是**动态加载**页面，数据通过 **API** 返回。

接着用浏览器的 `F12 → Network → Fetch/XHR` 刷新 `/devices`，找到那条 `api/devices` 请求，
记录下它的 Request URL、Method 与 Query 参数，然后在代码里复现。

In [3]:
# 复现 Network 中观察到的请求
api_url = BASE_URL + "/api/devices"

r = session.get(
    api_url,
    params={
        "page": 1,
        "page_size": 50,
    },
    timeout=10
)

print("status:", r.status_code)
print("url:", r.url)
print("content-type:", r.headers.get("Content-Type"))

status: 200
url: https://115.159.159.82/api/devices?page=1&page_size=50
content-type: application/json


In [4]:
data = r.json()

print(type(data))
print(data.keys())

<class 'dict'>
dict_keys(['page', 'page_size', 'total', 'items'])


In [5]:
# 不带 items 的部分就是分页元信息
print(json.dumps({k: v for k, v in data.items() if k != "items"}, ensure_ascii=False))
print("本页 items 条数:", len(data["items"]))

{"page": 1, "page_size": 50, "total": 5000}
本页 items 条数: 50


In [6]:
# 观察一条真实的设备记录，确认各字段含义
print(json.dumps(data["items"][0], ensure_ascii=False, indent=2))

{
  "device_id": "D00000001",
  "company_id": "C000001",
  "device_name": "循环泵-000001-001",
  "device_type": "pump",
  "model": "SP-PUM-800",
  "zone": "智造一区",
  "install_date": "2021-09-19",
  "rated_power_kw": 71.2,
  "sampling_interval_s": 30,
  "protocol": "mqtt",
  "status": "running",
  "updated_at": "2026-09-01T03:58:00"
}


## 1.2 API 信息与分页结构

在 Network 的 Fetch/XHR 中记录到的请求信息：

| 项目 | 内容 |
|---|---|
| Request URL | `https://115.159.159.82/api/devices?page=1&page_size=50` |
| Request Method | `GET` |
| Query 参数 | `page`（页码，从 1 开始）、`page_size`（每页条数，页面自带脚本用 50） |
| Content-Type | `application/json` |

从 JSON 中观察到的字段：

| 含义 | JSON 字段 |
|---|---|
| 当前页 | `page` |
| 每页条数 | `page_size` |
| 设备总数 | `total` |
| 设备列表 | `items` |

`items` 里每条设备记录的字段：

```text
device_id  company_id  device_name  device_type  model  zone
install_date  rated_power_kw  sampling_interval_s  protocol  status  updated_at
```

## 1.3 计算总页数

**不要写死页数**，用接口返回的 `total` 和 `page_size` 算出来。

In [7]:
total = data["total"]
page_size = data["page_size"]

total_pages = (total + page_size - 1) // page_size

print("total:", total)
print("page_size:", page_size)
print("total_pages:", total_pages)

total: 5000
page_size: 50
total_pages: 100


## 1.4 分页采集全部设备

In [8]:
devices = []

for page in range(1, total_pages + 1):
    r = session.get(
        api_url,
        params={
            "page": page,
            "page_size": page_size,
        },
        timeout=10
    )

    page_data = r.json()

    # 把本页设备加入总列表
    devices.extend(page_data["items"])

print("device count:", len(devices))

device count: 5000


In [9]:
# 设备记录字段清单（API 返回的结构化字段可直接保留）
print("设备字段：")
for k, v in devices[0].items():
    print(f"  {k:20s} {v}")

设备字段：
  device_id            D00000001
  company_id           C000001
  device_name          循环泵-000001-001
  device_type          pump
  model                SP-PUM-800
  zone                 智造一区
  install_date         2021-09-19
  rated_power_kw       71.2
  sampling_interval_s  30
  protocol             mqtt
  status               running
  updated_at           2026-09-01T03:58:00


# 任务 2：Company 静态页面解析

打开 `/companies`，同样先看“网页源代码”，搜索页面上真实显示的企业名称。

In [10]:
current_url = BASE_URL + "/companies"

r = session.get(current_url, timeout=10)
soup = BeautifulSoup(r.text, "html.parser")

print("status:", r.status_code)
print("title:", soup.title.get_text(strip=True))
print("初始 HTML 长度:", len(r.text))
print("源码中出现企业名（含“有限公司”）?", "有限公司" in r.text)
print("源码中出现 “临港精工001有限公司” ?", "临港精工001有限公司" in r.text)

status: 200
title: 园区企业 · SmartPark DataLab
初始 HTML 长度: 11331
源码中出现企业名（含“有限公司”）? True
源码中出现 “临港精工001有限公司” ? True


结论：企业记录**就在**初始 HTML 里，属于**静态加载**，可以直接 `requests → BeautifulSoup → CSS Selector`。

## 2.1 设计企业记录的 CSS Selector

一条完整企业记录的最外层节点是 `<article class="company-card" data-company-id="C000001">`。

按“id → class → data-* → 父子层级”的顺序考虑后，选择：

```css
article.company-card
```

理由：`class` 语义明确且全页复用（每家企业一个），比 `nth-child(...)` 之类的长路径稳定得多。

In [11]:
cards = soup.select("article.company-card")

print("本页企业数：", len(cards))
print("第一条外层标签:", cards[0].name, "| class:", cards[0].get("class"))

本页企业数： 20
第一条外层标签: article | class: ['company-card']


## 2.2 提取企业字段

先定位**一条**记录，再在这条记录**内部**提取字段。
行业 / 区域 / 员工规模 / 设备数量是 `dt/dd` 结构，可先转成字典。

In [12]:
card = cards[0]

details = {}

for item in card.select("dl > div"):
    key = item.select_one("dt").get_text(strip=True)
    value = item.select_one("dd").get_text(strip=True)
    details[key] = value

print(details)

{'所属行业': '汽车零部件', '所在区域': '智造一区', '员工规模': '88 人', '设备数量': '6 台'}


In [13]:
def parse_company(card):
    # 从一个 company-card 节点中提取企业字段
    details = {}
    for item in card.select("dl > div"):
        key = item.select_one("dt").get_text(strip=True)
        value = item.select_one("dd").get_text(strip=True)
        details[key] = value

    return {
        # 来自 HTML 属性 data-company-id
        "company_id": card.get("data-company-id"),
        # 来自文本：h2 > a
        "name": card.select_one("h2 a").get_text(strip=True),
        "industry": details.get("所属行业"),
        "zone": details.get("所在区域"),
        # dd 形如 “6 台”，取出数字再转 int
        "device_count": int(re.search(r"\d+", details.get("设备数量", "0")).group()),
        # 来自文本：状态徽标 span.badge
        "status": card.select_one("span.badge").get_text(strip=True),
    }


print(json.dumps(parse_company(cards[0]), ensure_ascii=False, indent=2))

{
  "company_id": "C000001",
  "name": "临港精工001有限公司",
  "industry": "汽车零部件",
  "zone": "智造一区",
  "device_count": 6,
  "status": "运营中"
}


In [14]:
# 第一页 20 条企业
page_companies = [parse_company(c) for c in cards]

print("本页企业数：", len(page_companies))
print(json.dumps(page_companies[0], ensure_ascii=False))

本页企业数： 20
{"company_id": "C000001", "name": "临港精工001有限公司", "industry": "汽车零部件", "zone": "智造一区", "device_count": 6, "status": "运营中"}


# 任务 3：Company “下一页”连续采集 3 页

页面底部有一个 `<nav class="pagination">`，其中的“下一页”是一个 `<a>` 元素：

```html
<nav class="pagination" aria-label="企业分页">
  <span>上一页</span><b>1</b><a href="/companies?page=2">下一页</a>
</nav>
```

它没有 id，也没有独立的 class，因此用**父子层级 + 位置**定位：`nav.pagination a:last-of-type`。
（比 `nth-child(...)` 长路径更短、更稳定；代码里再用文本 `== "下一页"` 兜底校验。）

In [15]:
next_link = soup.select_one("nav.pagination a:last-of-type")

print("标签:", next_link.name)
print("class:", next_link.get("class"))
print("文本:", next_link.get_text(" ", strip=True))
print("href:", next_link.get("href"))

next_url = urljoin(current_url, next_link.get("href"))
print("拼接后的绝对地址:", next_url)

标签: a
class: None
文本: 下一页
href: /companies?page=2
拼接后的绝对地址: https://115.159.159.82/companies?page=2


## 3.1 三页循环

把「请求当前页 → 解析本页企业 → 找到下一页 href」组合进循环，连续采集 3 页。

In [16]:
companies = []
current_url = BASE_URL + "/companies"

for page_no in range(1, 4):
    r = session.get(current_url, timeout=10)
    soup = BeautifulSoup(r.text, "html.parser")

    cards = soup.select("article.company-card")
    page_companies = [parse_company(c) for c in cards]
    companies.extend(page_companies)

    print(f"第 {page_no} 页：{len(page_companies)} 条   (URL: {current_url})")

    # 当前页 → 找下一页 href
    next_link = soup.select_one("nav.pagination a:last-of-type")
    if next_link is None or next_link.get_text(strip=True) != "下一页":
        print("  已到最后一页或没有“下一页”，停止")
        break

    # 下一页 → 作为下一轮的请求地址
    current_url = urljoin(current_url, next_link.get("href"))

print()
print("Company 总数：", len(companies))

第 1 页：20 条   (URL: https://115.159.159.82/companies)
第 2 页：20 条   (URL: https://115.159.159.82/companies?page=2)
第 3 页：20 条   (URL: https://115.159.159.82/companies?page=3)

Company 总数： 60


# 任务 4：验证采集结果

In [17]:
print(f"Company 总数: {len(companies)}")
print(f"唯一 company_id: {len(set(x['company_id'] for x in companies))}")
print(f"Device 总数: {len(devices)}")
print(f"唯一 device_id: {len(set(x['device_id'] for x in devices))}")

Company 总数: 60
唯一 company_id: 60
Device 总数: 5000
唯一 device_id: 5000


In [18]:
device_counts = Counter(x["company_id"] for x in devices)

mismatch_count = 0
mismatch_examples = []

for company in companies:
    cid = company["company_id"]
    expected = company["device_count"]          # 企业页面上的设备数量
    actual = device_counts.get(cid, 0)          # 由 Device 数据统计出的实际设备数

    if expected != actual:
        mismatch_count += 1
        mismatch_examples.append((cid, expected, actual))

print("Company–Device 不一致企业数：", mismatch_count)
print("示例：", mismatch_examples[:5])

Company–Device 不一致企业数： 0
示例： []


In [19]:
# 便于截图：把 5 个校验指标放在同一个单元格输出
print(f"Company 总数: {len(companies)}")
print(f"唯一 company_id: {len(set(x['company_id'] for x in companies))}")
print(f"Device 总数: {len(devices)}")
print(f"唯一 device_id: {len(set(x['device_id'] for x in devices))}")
print(f"Company–Device 不一致企业数: {mismatch_count}")

Company 总数: 60
唯一 company_id: 60
Device 总数: 5000
唯一 device_id: 5000
Company–Device 不一致企业数: 0


# 任务 5：保存采集结果

In [20]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

with open(data_dir / "devices.json", "w", encoding="utf-8") as f:
    json.dump(devices, f, ensure_ascii=False, indent=2)

with open(data_dir / "companies.json", "w", encoding="utf-8") as f:
    json.dump(companies, f, ensure_ascii=False, indent=2)

print("数据已保存到 data/ 目录")
print(" -", (data_dir / "devices.json").resolve())
print(" -", (data_dir / "companies.json").resolve())

数据已保存到 data/ 目录
 - C:\Users\wang\Desktop\lab01\data\devices.json
 - C:\Users\wang\Desktop\lab01\data\companies.json


---

# 完成前自检

```text
☑ 会用“查看网页源代码”判断静态 / 动态加载
☑ 能在 Network 中找到 Device API
☑ 能读懂 API 分页结构并自动计算总页数
☑ Device 共采集 5000 条，device_id 无重复
☑ 能设计简洁、稳定的 Company CSS Selector
☑ 能定位“下一页”并读取 href
☑ Company 连续采集 3 页，共 60 条
☑ company_id 无重复
☑ Company–Device 数量检查通过
☑ 已保存 companies.json 和 devices.json
```